# Retrieval — Hybrid Search + Cross-Encoder Reranking

Dense vector search alone misses queries that rely on exact terms — ticker symbols, dollar figures, specific product names. BM25 alone misses queries that rely on meaning rather than keywords. Combining both and then reranking with a cross-encoder gives the best of all three.

The pipeline this notebook builds:

```
Query
  ├─► Dense retrieval   (ChromaDB + bi-encoder)  → top-50 candidates
  ├─► BM25 retrieval    (rank_bm25)               → top-50 candidates
  ├─► RRF fusion        (Reciprocal Rank Fusion)  → top-20 merged
  └─► Cross-encoder     (rerank merged list)      → top-5 final results
```

**Why each step?**

| Step | What it adds |
|---|---|
| Dense | Semantic similarity — finds paraphrases and related concepts |
| BM25 | Exact keyword match — reliable for tickers, numbers, proper nouns |
| RRF | Merges rankings without needing to normalise incompatible scores |
| Cross-encoder | Reads query + passage together — far more accurate than bi-encoder alone, but too slow to run over the full corpus |

## 1. Setup

In [ ]:
import sys
import pickle
from pathlib import Path

import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

sys.path.insert(0, str(Path("..").resolve()))
from utils import Chunk, get_chroma_client

# ── Retrieval constants ───────────────────────────────────────────────────────

# How many candidates each retriever returns before fusion.
# Casting a wide net here is cheap — the expensive cross-encoder only sees TOP_RERANK.
TOP_K_DENSE  = 50
TOP_K_BM25   = 50

# How many fused candidates to pass to the cross-encoder.
TOP_RERANK   = 20

# Final results returned to the caller.
TOP_K_FINAL  = 5

# RRF smoothing constant — 60 is the standard value from the original paper.
RRF_K        = 60

CHROMA_DIR   = "../chroma_db"
COLLECTION   = "finance_rag"

print("Setup complete.")

## 2. Load Chunks & Indexes

Both retrievers need the full chunk list in memory:
- **ChromaDB** uses it implicitly (vectors are stored on disk)
- **BM25** is an in-memory index we build at runtime — fast to construct and doesn't need to be persisted

In [ ]:
cache_path = Path("../chunks_cache.pkl")

if not cache_path.exists():
    raise FileNotFoundError(
        f"Cache not found at {cache_path}. "
        "Run 01_parsing_strategy.ipynb first."
    )

with open(cache_path, "rb") as f:
    all_chunks = pickle.load(f)

print(f"Loaded {len(all_chunks)} chunks")

In [ ]:
# Connect to the ChromaDB collection populated by 02_embeddings.ipynb.
client     = get_chroma_client(CHROMA_DIR)
collection = client.get_collection(COLLECTION)

print(f"Collection '{COLLECTION}': {collection.count()} documents")

In [ ]:
# Load the bi-encoder used during indexing — query vectors must live in the
# same embedding space as the stored document vectors.
bi_encoder = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Bi-encoder loaded  (dim={bi_encoder.get_sentence_embedding_dimension()})")

# Build the BM25 index over all chunk texts.
# Simple whitespace tokenisation works well for financial text — tickers and
# figures we most want to match are single tokens after splitting.
tokenized_corpus = [c.text.lower().split() for c in all_chunks]
bm25 = BM25Okapi(tokenized_corpus)

print(f"BM25 index built   ({len(tokenized_corpus)} documents)")

## 3. Dense Retrieval

The bi-encoder embeds the query into the same vector space as the stored chunks. ChromaDB returns the nearest neighbours by L2 distance.

In [ ]:
def dense_retrieve(query: str, k: int = TOP_K_DENSE) -> list[tuple[int, float]]:
    """
    Returns (chunk_index, score) pairs sorted by relevance descending.
    Score is converted from L2 distance to 0-1 similarity so higher = better.
    """
    query_vec = bi_encoder.encode(query, convert_to_numpy=True).tolist()

    # IDs are always returned by ChromaDB — they must NOT be listed in include.
    results = collection.query(
        query_embeddings=[query_vec],
        n_results=k,
        include=["distances"],
    )

    # ChromaDB returns string IDs — map back to list indices.
    id_to_idx = {c.id: i for i, c in enumerate(all_chunks)}

    hits = []
    for cid, dist in zip(results["ids"][0], results["distances"][0]):
        idx = id_to_idx.get(cid)
        if idx is not None:
            hits.append((idx, 1 / (1 + dist)))  # convert distance to similarity

    return hits  # already sorted best-first by ChromaDB


# ── Quick test ────────────────────────────────────────────────────────────────
TEST_QUERY = "What was Amazon's net revenue in 2024?"

dense_hits = dense_retrieve(TEST_QUERY)
print(f"Dense top-5 for: '{TEST_QUERY}'\n")
for rank, (idx, score) in enumerate(dense_hits[:5], 1):
    c = all_chunks[idx]
    print(f"  {rank}. [{score:.3f}] {c.source} p{c.page} — {c.text[:80].replace(chr(10),' ')}...")

## 4. BM25 Sparse Retrieval

BM25 scores documents by term frequency weighted against how rare each term is across the corpus (IDF). It reliably surfaces chunks that contain the exact words in the query — especially useful for financial figures, ticker symbols, and named entities that a semantic model might miss.

In [ ]:
def bm25_retrieve(query: str, k: int = TOP_K_BM25) -> list[tuple[int, float]]:
    """
    Returns (chunk_index, score) pairs sorted by BM25 score descending.
    """
    tokens = query.lower().split()
    scores = bm25.get_scores(tokens)

    # argsort is ascending — flip it and take top-k.
    top_indices = scores.argsort()[::-1][:k]

    return [(int(i), float(scores[i])) for i in top_indices]


# ── Quick test ────────────────────────────────────────────────────────────────
bm25_hits = bm25_retrieve(TEST_QUERY)
print(f"BM25 top-5 for: '{TEST_QUERY}'\n")
for rank, (idx, score) in enumerate(bm25_hits[:5], 1):
    c = all_chunks[idx]
    print(f"  {rank}. [{score:.3f}] {c.source} p{c.page} — {c.text[:80].replace(chr(10),' ')}...")

## 5. Reciprocal Rank Fusion (RRF)

RRF merges two ranked lists without normalising their scores (BM25 and cosine similarity live on completely different scales).

Each document gets a fusion score based on its rank in each list:

```
score(d) = sum over lists:  1 / (k + rank(d))
```

A document ranked 1st in both lists scores higher than one ranked 1st in only one. Documents absent from a list simply contribute nothing from that term.

In [ ]:
def reciprocal_rank_fusion(
    *ranked_lists: list[tuple[int, float]],
    k: int = RRF_K,
    top_n: int = TOP_RERANK,
) -> list[int]:
    """
    Merge any number of (chunk_index, score) ranked lists via RRF.
    Returns chunk indices sorted by fused score descending.
    """
    fused: dict[int, float] = {}

    for ranked in ranked_lists:
        for rank, (idx, _) in enumerate(ranked):
            fused[idx] = fused.get(idx, 0.0) + 1.0 / (k + rank + 1)

    sorted_ids = sorted(fused, key=fused.__getitem__, reverse=True)
    return sorted_ids[:top_n]


# ── Quick test ────────────────────────────────────────────────────────────────
fused_ids = reciprocal_rank_fusion(dense_hits, bm25_hits)
print(f"Fused top-5 for: '{TEST_QUERY}'\n")
for rank, idx in enumerate(fused_ids[:5], 1):
    c = all_chunks[idx]
    print(f"  {rank}. {c.source} p{c.page} — {c.text[:80].replace(chr(10),' ')}...")

## 6. Cross-Encoder Reranking

A bi-encoder embeds query and document *independently* — it never directly compares them. A cross-encoder reads both together as a single input and produces a fine-grained relevance score. Much more accurate, but too slow to run over thousands of documents.

The pattern: use the fast retrievers to narrow to ~20 candidates, then let the cross-encoder pick the best 5.

In [ ]:
# ms-marco-MiniLM-L-6-v2 is trained on MS MARCO passage ranking.
# It outputs a raw logit — higher is more relevant.
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Cross-encoder loaded.")

In [ ]:
def rerank(
    query: str,
    candidate_ids: list[int],
    top_n: int = TOP_K_FINAL,
) -> list[tuple[int, float]]:
    """
    Score each (query, chunk) pair with the cross-encoder.
    Returns (chunk_index, score) sorted best-first.
    """
    pairs = [(query, all_chunks[i].text) for i in candidate_ids]

    # predict() returns a numpy array of raw relevance logits.
    scores = cross_encoder.predict(pairs, show_progress_bar=False)

    ranked = sorted(
        zip(candidate_ids, scores.tolist()),
        key=lambda x: x[1],
        reverse=True,
    )
    return ranked[:top_n]


# ── Quick test ────────────────────────────────────────────────────────────────
reranked = rerank(TEST_QUERY, fused_ids)
print(f"Reranked top-5 for: '{TEST_QUERY}'\n")
for rank, (idx, score) in enumerate(reranked, 1):
    c = all_chunks[idx]
    print(f"  {rank}. [{score:.3f}] {c.source} p{c.page} — {c.text[:80].replace(chr(10),' ')}...")

## 7. Full Pipeline

`retrieve()` wraps the four steps into a single call. This is what the RAG layer will call at query time.

In [ ]:
def retrieve(query: str, top_k: int = TOP_K_FINAL) -> list[Chunk]:
    """
    Full hybrid retrieval pipeline:
      1. Dense search  (bi-encoder + ChromaDB)
      2. Sparse search (BM25)
      3. RRF fusion
      4. Cross-encoder reranking
    Returns the top_k most relevant Chunk objects.
    """
    d_hits  = dense_retrieve(query,  k=TOP_K_DENSE)
    s_hits  = bm25_retrieve(query,   k=TOP_K_BM25)
    fused   = reciprocal_rank_fusion(d_hits, s_hits, top_n=TOP_RERANK)
    ranked  = rerank(query, fused, top_n=top_k)

    return [all_chunks[idx] for idx, _ in ranked]


print("retrieve() defined.")

## 8. Compare Strategies

Run the same query through each stage to see how results shift as we add each layer. Differences in the final list compared to dense-only show where sparse signals and reranking are earning their compute cost.

In [ ]:
COMPARE_QUERY = "What was Amazon's net revenue in 2024?"

d_hits = dense_retrieve(COMPARE_QUERY)
s_hits = bm25_retrieve(COMPARE_QUERY)
fused  = reciprocal_rank_fusion(d_hits, s_hits)

strategies = {
    "Dense only":      [all_chunks[i] for i, _ in d_hits[:5]],
    "BM25 only":       [all_chunks[i] for i, _ in s_hits[:5]],
    "Hybrid (RRF)":    [all_chunks[i] for i in fused[:5]],
    "Hybrid + Rerank": retrieve(COMPARE_QUERY),
}

for label, chunks in strategies.items():
    print(f"\n{'─'*65}")
    print(f" {label}")
    print(f"{'─'*65}")
    for rank, c in enumerate(chunks, 1):
        preview = c.text[:90].replace("\n", " ")
        print(f"  {rank}. [{c.document_type:<12}] {c.source[:35]:<35}  {preview}...")